# Integrated syntactic graph (ISG) vectors of every text

Builds the graph of every text of the three splits once and saves the vectors of both texts of every
pair in one folder; the two ISG notebooks load it. The folder is accepted only with the same graph.py,
uv.lock, data and settings, so the ISG notebooks can run at a later commit.

The build does not resume - a restarted run removes the half-built folder and builds again. A complete
folder is not built again; the checks at the end run on it either way.

In [ ]:
SMOKE = False  # True - the first rows of every data set, a folder of its own
RUN_TAG = ""  # label of a smoke folder, so two smoke rounds never share one
WORKERS = 16

Standard libraries

In [ ]:
import hashlib
import json
import logging
import re
import shutil
import tempfile
from itertools import islice
from pathlib import Path

Numbers

In [ ]:
import numpy as np

The shared modules of the thesis

In [ ]:
from avcore import data, features, graph, progress, tracking

# Configuration

In [ ]:
SMOKE_ROWS = {"train": 120, "validation": 40, "test": 40}

The data files

In [ ]:
# the notebook runs from its own folder, two levels below the experiments folder
PROJECT_DIR = Path.cwd().parents[1]
DATA_DIR = PROJECT_DIR / "data"
CLEANED_PATHS = {
    "train": DATA_DIR / "processed" / data.TRAIN_CLEANED_FILE,
    "validation": DATA_DIR / "processed" / data.VALIDATION_CLEANED_FILE,
    "test": DATA_DIR / "processed" / data.TEST_CLEANED_FILE,
}

The ISG vector folder, outside every checkout

In [ ]:
VECTORS_DIR = Path.home() / "isg_vectors_v2"
if SMOKE:
    VECTORS_DIR = Path.home() / "isg_vectors_v2_smoke" / str(RUN_TAG)

The build report - counts of pairs, columns and empty graphs, in the git-ignored outputs

In [ ]:
REPORT_DIR = PROJECT_DIR / "outputs" / "isg_vector_extraction"
DEVELOPER_INITIALS = "JP"

Smoke run - copies of the first lines of every file

In [ ]:
def copy_first_lines(path: Path, n_lines: int, target_dir: Path) -> Path:
    """Copy the first n_lines of a file into target_dir, under the same file name."""
    with path.open(encoding="utf-8") as file:
        first_lines = list(islice(file, n_lines))
    copy_path = target_dir / path.name
    copy_path.write_text("".join(first_lines), encoding="utf-8")
    return copy_path

In [ ]:
temp_dir = tempfile.TemporaryDirectory()
if SMOKE:
    files_to_read = {}
    for split, path in CLEANED_PATHS.items():
        files_to_read[split] = copy_first_lines(
            path,
            SMOKE_ROWS[split],
            Path(temp_dir.name),
        )
else:
    files_to_read = CLEANED_PATHS

Run context - commit, hashes and versions; refuses uncommitted or unpushed changes

In [ ]:
logging.basicConfig(format="%(asctime)s %(message)s")
run_context = tracking.collect_run_context(PROJECT_DIR, list(files_to_read.values()))
print(f"Run {RUN_NAME}, commit {run_context.environment['commit'][:7]}")
progress_file = progress.ProgressFile(
    Path.home() / "logs" / f"{RUN_NAME}.progress.json",
    RUN_NAME,
)